In [ ]:
#!/usr/bin/env python3
"""Interactive explorer of the UMAP/t-SNE layouts saved by npz_scanpy.py: joint or per-species views in 2D/3D, colouring by saved clusters, live HDBSCAN, manual clusters and exports.

Every .h5ad found one level below the npz_scanpy outdir of configs/param_configs.yaml is listed, and only the views (scope x dimension) whose layouts exist in it are offered, with one panel per available method. Species are shown with the exact_species names of the query configuration and coloured in its order. In 2D points are selected with lasso/box; in 3D clicks add points to the selection.
"""
###################################### DEPENDENCIES_AND_CONSTANTS
import anndata as ad
import datetime
import hdbscan
from collections.abc import Callable
from IPython.display import clear_output, display
from ipywidgets import Button, Checkbox, ColorPicker, Dropdown, HBox, IntSlider, Layout, Output, Text, VBox
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyprojroot import find_root, has_file
import resource
import time
import traceback
import yaml
ALPHA = 0.75
ALPHA_INACTIVE = 0.03
COLOR_BY = ["(manual)", "species", "cluster", "intra_cluster"]
DEFAULT_COLOR = "#BFC5CE"
DEFAULT_OUTDIR = "data/scanpy/"
DEFAULT_QUERY_TSV = "configs/query_configs.tsv"
DIMS = (2, 3)
DPI = 300
EXPORT_DIR = "viz/dimred"
FIGURE_HEIGHT = 680
FIGURE_WIDTH = 1200
FONT_FAMILY = "DejaVu Sans"
FONT_LEGEND = 13
FONT_TICK = 12
FONT_TITLE = 16
LEGEND_NCOL_MAX = 5
LINE_COLOR = "black"
LINE_WIDTH = 0.8
MARKER_SIZE = 3
MARKER_SIZE_3D = 2
MAX_CLUSTERS = 15
METHODS = {"umap": "UMAP", "tsne": "t-SNE"}
MIN_CLUSTER_SIZE = 800
MIN_SAMPLES = 10
PALETTE = ["#ECDE90", "#93DEB6", "#F1B584", "#76BBE2", "#DA919B", "#899E64", "#8E81B9", "#A66C58", "#30767A", "#7A5271"]
SEED = 42
SQUARE_SIZE = 9.0
YAML_PATH = "configs/param_configs.yaml"
ZOOM = {"umap": 1.05, "tsne": 1.05}
###################################### DISCOVERY
start_time = time.time()
PROJECT_ROOT = find_root(has_file(".here"), start=Path.cwd()).resolve()
YDATA = (yaml.safe_load((PROJECT_ROOT / YAML_PATH).read_text()) or {}) if (PROJECT_ROOT / YAML_PATH).exists() else {}
def species_names() -> dict[str, str]:
    """Map each label of the query configuration to its exact_species name, keeping the configuration order.

    Returns:
        dict[str, str]: Label -> display name.
    """
    tsv = PROJECT_ROOT / YDATA.get("dw_uniprot_seqs", {}).get("config_path", DEFAULT_QUERY_TSV)
    df = pd.read_csv(tsv, sep="\t", dtype=str) if tsv.exists() else pd.DataFrame(columns=["label"])
    names = df["exact_species"] if "exact_species" in df else df["label"]
    return {lbl: (nm if pd.notna(nm) and str(nm).strip() else lbl) for lbl, nm in zip(df["label"], names) if pd.notna(lbl)}
SPECIES_NAMES = species_names()
def species_order(labels: list[str]) -> list[str]:
    """Order species labels as in the query configuration, unknown ones last.

    Args:
        labels (list[str]): Species labels.

    Returns:
        list[str]: Ordered labels.
    """
    rank = {s: i for i, s in enumerate(SPECIES_NAMES)}
    return sorted(dict.fromkeys(labels), key=lambda s: (rank.get(s, len(rank)), s))
def discover_views(outdir: Path) -> dict[str, tuple[Path, str, int, tuple[str, ...]]]:
    """List the available views, reading only obs and the obsm keys of each .h5ad.

    Args:
        outdir (Path): npz_scanpy output directory.

    Returns:
        dict[str, tuple[Path, str, int, tuple[str, ...]]]: Label -> (h5ad path, scope ``joint`` or species, dimension, methods).
    """
    views = {}
    for path in sorted(outdir.glob("*/*.h5ad")):
        adata = ad.read_h5ad(path, backed="r")
        keys, obs, run = set(adata.obsm.keys()), adata.obs, path.parent.name
        for d in DIMS:
            joint = tuple(m for m in METHODS if f"X_{m}_{d}d" in keys)
            if joint: views[f"{run} | all species | {d}D"] = (path, "joint", d, joint)
            intra = tuple(m for m in METHODS if f"X_{m}_{d}d_intra" in keys)
            for sp in species_order(list(obs["species"].astype(str).unique())) if intra else []:
                if obs.loc[obs["species"] == sp, "intra_cluster"].notna().any(): views[f"{run} | {SPECIES_NAMES.get(sp, sp)} | {d}D"] = (path, sp, d, intra)
        adata.file.close()
    return views
OUTDIR = PROJECT_ROOT / YDATA.get("npz_scanpy", {}).get("outdir", DEFAULT_OUTDIR)
VIEWS = discover_views(OUTDIR)
if not VIEWS: raise FileNotFoundError(f"[PYTHON-ERROR] No .h5ad with UMAP/t-SNE layouts found in {OUTDIR}/<run>/; run slurms/supp_01_scanpy.sh first.")
SPECIES_COLORS = {SPECIES_NAMES.get(s, s): PALETTE[i % len(PALETTE)] for i, s in enumerate(species_order(list(SPECIES_NAMES)))}
CACHE = {}
print(f"[PYTHON-INFO] {len(VIEWS)} views found in {OUTDIR}")
print(f"[PYTHON-INFO] Species: {', '.join(SPECIES_NAMES.values()) or 'none in query configuration'}")

In [ ]:
###################################### WIDGETS
select_dropdown = Dropdown(options=[""] + list(VIEWS), description="View:", layout=Layout(width="460px"))
color_by_dropdown = Dropdown(options=COLOR_BY, value="(manual)", description="Color by:", layout=Layout(width="240px"))
reset_angles_btn = Button(description="Reset view", button_style="warning")
checklist_box = HBox([])
mcs_slider = IntSlider(value=MIN_CLUSTER_SIZE, min=5, max=10000, step=5, description="MinCS:", continuous_update=False, layout=Layout(width="340px"))
ms_slider = IntSlider(value=MIN_SAMPLES, min=1, max=200, step=1, description="MinSamples:", continuous_update=False, layout=Layout(width="300px"), style={"description_width": "initial"})
fit_on_dropdown = Dropdown(options=list(METHODS), value="umap", description="Fit on:", layout=Layout(width="180px"))
fit_btn = Button(description="Fit HDBSCAN", button_style="primary")
cluster_name = Text(value="NewCluster", description="Name:", layout=Layout(width="210px"))
cluster_color = ColorPicker(value="#7b2c6d", description="Color:")
assign_btn = Button(description="Assign selected", button_style="primary")
clear_sel_btn = Button(description="Clear selection", button_style="warning")
rename_dropdown = Dropdown(options=[], description="Rename:", layout=Layout(width="210px"))
rename_text = Text(value="", description="To:", layout=Layout(width="160px"))
rename_btn = Button(description="Rename", button_style="info")
reset_btn = Button(description="Reset clusters", button_style="danger")
delete_btn = Button(description="Delete cluster", button_style="danger")
plot_fp_text = Text(value=EXPORT_DIR, description="Export dir:", layout=Layout(width="340px"))
export_tsv_btn = Button(description="Export TSV", button_style="info")
export_plot_btn = Button(description="Export Plot", button_style="info")
export_html_btn = Button(description="Export HTML", button_style="info")
run_all_btn = Button(description="Run All", button_style="success")
overwrite_cb = Checkbox(value=False, description="Overwrite", indent=False, layout=Layout(width="100px"))
output_area = Output()
plot_container = VBox([])
state = {"ids": None, "coords": {}, "methods": (), "obs": None, "cluster_labels": None, "cluster_info": {}, "filtered_mask": None, "current": None, "label_source": None, "checkboxes": {}, "fig_widget": None, "selected_indices": [], "last_selection_source": "umap", "display_title": "", "dim": 2}
###################################### DATA_AND_CLUSTERING
def load_view(label: str) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    """Load the proteins and layouts of a view, caching each .h5ad once.

    Args:
        label (str): View label from ``VIEWS``.

    Returns:
        tuple[pd.DataFrame, dict[str, np.ndarray]]: Observations of the view and layout per method.
    """
    path, scope, d, methods = VIEWS[label]
    if path not in CACHE: CACHE[path] = ad.read_h5ad(path)
    adata = CACHE[path]
    mask = np.ones(adata.n_obs, dtype=bool) if scope == "joint" else (adata.obs["species"] == scope).values
    suffix = "" if scope == "joint" else "_intra"
    return adata.obs[mask].reset_index(drop=True), {m: np.asarray(adata.obsm[f"X_{m}_{d}d{suffix}"][mask], dtype=np.float64) for m in methods}
def view_tag(label: str) -> str:
    """Turn a view label into a file-name tag.

    Args:
        label (str): View label.

    Returns:
        str: Tag such as ``run_joint_2D``.
    """
    path, scope, d, _ = VIEWS[label]
    return f"{path.parent.name}_{scope}_{d}D"
def fit_hdbscan_auto(coords: np.ndarray, min_cluster_size: int, min_samples: int) -> tuple[np.ndarray, int]:
    """Fit HDBSCAN and relabel the MAX_CLUSTERS largest clusters 1..n by decreasing size, the rest 0.

    Args:
        coords (np.ndarray): Layout coordinates.
        min_cluster_size (int): HDBSCAN minimum cluster size.
        min_samples (int): HDBSCAN min_samples, capped at min_cluster_size.

    Returns:
        tuple[np.ndarray, int]: Labels and number of kept clusters.
    """
    mcs = int(min_cluster_size)
    if len(coords) <= mcs: raise ValueError(f"Only {len(coords)} points available; MinCS is {mcs}")
    raw = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=min(int(min_samples), mcs), metric="euclidean").fit_predict(coords)
    valid = raw[raw >= 0]
    if not len(valid): return np.zeros(len(coords), dtype=int), 0
    unique, counts = np.unique(valid, return_counts=True)
    mapping = {int(old): rank + 1 for rank, old in enumerate(unique[np.argsort(-counts, kind="stable")][:MAX_CLUSTERS])}
    return np.asarray([mapping.get(int(x), 0) for x in raw], dtype=int), len(mapping)
def palette_info(names: list[str], colors: list[str] | None = None) -> dict[int, dict[str, str]]:
    """Build the cluster table with alternating PALETTE colours (or the given ones), id 0 being unassigned.

    Args:
        names (list[str]): Names of clusters 1..n.
        colors (list[str] | None): Colours of clusters 1..n.

    Returns:
        dict[int, dict[str, str]]: Cluster id -> name and colour.
    """
    colors = colors or [PALETTE[i % len(PALETTE)] for i in range(len(names))]
    return {0: {"name": "unassigned", "color": DEFAULT_COLOR}} | {i + 1: {"name": n, "color": c} for i, (n, c) in enumerate(zip(names, colors))}
def next_cluster_id() -> int:
    """Return the next free manual cluster id."""
    return max((int(cid) for cid in state["cluster_info"] if int(cid) != 0), default=0) + 1
def cluster_color_for(cid: int) -> str:
    """Return the colour of a cluster id."""
    return state["cluster_info"].get(int(cid), {"color": DEFAULT_COLOR})["color"]
def cluster_name_for(cid: int) -> str:
    """Return the name of a cluster id."""
    return state["cluster_info"].get(int(cid), {"name": "unassigned"})["name"]
###################################### PLOT_REFRESH
def update_checklist(labels: list[str]) -> None:
    """Rebuild the species visibility checkboxes.

    Args:
        labels (list[str]): Species display names of the current view.
    """
    state["checkboxes"] = {}
    for label in labels:
        cb = Checkbox(value=True, description=label, indent=False, layout=Layout(width="auto"))
        cb.observe(lambda change: update_plot_traces(), names="value")
        state["checkboxes"][label] = cb
    checklist_box.children = list(state["checkboxes"].values()) if len(labels) > 1 else []
def refresh_cluster_dropdown() -> None:
    """Refresh the rename/delete dropdown from the cluster table."""
    options = [f"{cid}: {info['name']}" for cid, info in sorted(state["cluster_info"].items()) if cid != 0]
    rename_dropdown.options = options
    rename_dropdown.value = options[0] if options else None
def rgba(color: str, alpha: float) -> str:
    """Convert a hex colour and an alpha to an rgba string.

    Args:
        color (str): Hex colour.
        alpha (float): Opacity.

    Returns:
        str: ``rgba(r,g,b,a)``.
    """
    return f"rgba({int(color[1:3], 16)},{int(color[3:5], 16)},{int(color[5:7], 16)},{alpha})"
def update_plot_traces() -> None:
    """Recolour the data traces, fade hidden species and rebuild the legend traces."""
    fw, labels, sources = state["fig_widget"], state["cluster_labels"], state["label_source"]
    if fw is None or labels is None: return
    n_data = len(state["methods"])
    visible = {label for label, cb in state["checkboxes"].items() if cb.value}
    mask = np.isin(sources, list(visible)) if state["checkboxes"] else np.ones(len(labels), dtype=bool)
    mask[np.asarray(state["selected_indices"], dtype=int)] = True
    state["filtered_mask"] = mask
    colors, alphas = [cluster_color_for(x) for x in labels], np.where(mask, ALPHA, ALPHA_INACTIVE)
    selected = np.zeros(len(labels), dtype=bool)
    selected[np.asarray(state["selected_indices"], dtype=int)] = True
    hover = [f"<b>{uid}</b><br>{src}<br>{cluster_name_for(cid)}" for src, cid, uid in zip(sources, labels, state["ids"])]
    scatter = go.Scatter3d if state["dim"] == 3 else go.Scatter
    empty = dict(x=[None], y=[None], z=[None]) if state["dim"] == 3 else dict(x=[None], y=[None])
    with fw.batch_update():
        for i in range(n_data):
            fw.data[i].text = hover
            if state["dim"] == 3:
                fw.data[i].marker.color = [rgba("#000000", 1.0) if s else rgba(c, a) for c, a, s in zip(colors, alphas, selected)]
            else:
                fw.data[i].marker.color, fw.data[i].marker.opacity = colors, alphas
        fw.data = tuple(fw.data[:n_data])
        for cid in sorted((x for x in state["cluster_info"] if x != 0), key=lambda x: -int(np.sum(labels == x))):
            info, count = state["cluster_info"][cid], int(np.sum(labels == cid))
            if count: fw.add_trace(scatter(**empty, mode="markers", marker=dict(size=10, color=info["color"]), name=f"{info['name']} (n={count:,})"))
        if np.any(labels == 0): fw.add_trace(scatter(**empty, mode="markers", marker=dict(size=10, color=DEFAULT_COLOR), name=f"unassigned (n={int(np.sum(labels == 0)):,})"))
    refresh_cluster_dropdown()
###################################### SELECTION_AND_EDITING
def selection_callback(source: str) -> Callable:
    """Build a lasso/box callback replacing the selection, and a click callback that adds to it in 3D.

    Args:
        source (str): Method of the trace (``umap`` or ``tsne``).

    Returns:
        Callable: Plotly trace callback.
    """
    def callback(trace: object, points: object, selector: object) -> None:
        """Store the selected indices of the trace."""
        indices = {int(i) for i in (getattr(points, "point_inds", []) or []) if 0 <= int(i) < len(state["ids"])}
        if not indices: return
        state["selected_indices"] = sorted(indices | (set(state["selected_indices"]) if state["dim"] == 3 else set()))
        state["last_selection_source"] = source
        if state["dim"] == 3: update_plot_traces()
        with output_area:
            clear_output(wait=True)
            print(f"[PYTHON-INFO] {len(state['selected_indices']):,} points selected from {METHODS[source]}.")
    return callback
def clear_selection(_: object = None) -> None:
    """Empty the current selection."""
    state["selected_indices"] = []
    update_plot_traces()
    with output_area:
        clear_output(wait=True)
        print("[PYTHON-INFO] Selection cleared.")
def assign_selected(_: object = None) -> None:
    """Assign the selected points to a new manual cluster."""
    with output_area:
        clear_output(wait=True)
        indices = state["selected_indices"]
        if not indices or state["cluster_labels"] is None:
            print("[PYTHON-INFO] Select points in a plot first (lasso/box in 2D, clicks in 3D).")
            return
        cid = next_cluster_id()
        name = cluster_name.value.strip()
        if not name or name == "NewCluster": name = f"{'U' if state['last_selection_source'] == 'umap' else 'T'}-{cid:02d}"
        state["cluster_labels"][indices] = cid
        state["cluster_info"][cid] = {"name": name, "color": cluster_color.value}
        state["selected_indices"] = []
        update_plot_traces()
        print(f"[PYTHON-INFO] Assigned {len(indices):,} points to cluster {cid} '{name}'.")
def rename_cluster(_: object = None) -> None:
    """Rename the cluster chosen in the dropdown."""
    with output_area:
        clear_output(wait=True)
        if not rename_dropdown.value or not rename_text.value.strip():
            print("[PYTHON-INFO] Choose a cluster and enter a new name.")
            return
        cid = int(rename_dropdown.value.split(":", 1)[0])
        state["cluster_info"][cid]["name"] = rename_text.value.strip()
        update_plot_traces()
        print(f"[PYTHON-INFO] Renamed cluster {cid} to '{rename_text.value.strip()}'.")
def reset_clusters(_: object = None) -> None:
    """Set every point back to unassigned."""
    with output_area:
        clear_output(wait=True)
        if state["ids"] is None:
            print("[PYTHON-INFO] Load a view first.")
            return
        state["cluster_labels"], state["cluster_info"], state["selected_indices"] = np.zeros(len(state["ids"]), dtype=int), palette_info([]), []
        color_by_dropdown.value = "(manual)"
        update_plot_traces()
        print("[PYTHON-INFO] All points reset to unassigned.")
def delete_cluster(_: object = None) -> None:
    """Delete the cluster chosen in the dropdown, unassigning its points."""
    with output_area:
        clear_output(wait=True)
        if not rename_dropdown.value:
            print("[PYTHON-INFO] Choose a cluster first.")
            return
        cid = int(rename_dropdown.value.split(":", 1)[0])
        state["cluster_labels"][state["cluster_labels"] == cid] = 0
        del state["cluster_info"][cid]
        update_plot_traces()
        print(f"[PYTHON-INFO] Deleted cluster {cid}; its points are now unassigned.")
def reset_angles(_: object = None) -> None:
    """Reset the 3D cameras or the 2D axis ranges."""
    fw = state["fig_widget"]
    if fw is None: return
    if state["dim"] == 3:
        fw.update_layout({f"scene{'' if i == 1 else i}_camera": dict(eye=dict(x=1.5, y=1.5, z=1.2)) for i in range(1, len(state["methods"]) + 1)})
    else:
        fw.update_xaxes(autorange=True)
        fw.update_yaxes(autorange=True)
###################################### LOADING_AND_PLOTTING
def zoom_range(values: np.ndarray, zoom: float) -> list[float]:
    """Return an axis range centred on the data and scaled by ``zoom``.

    Args:
        values (np.ndarray): Coordinates along one axis.
        zoom (float): Range multiplier.

    Returns:
        list[float]: Lower and upper bounds.
    """
    lo, hi = float(values.min()), float(values.max())
    mid, half = (lo + hi) / 2, (hi - lo) * zoom / 2
    return [mid - half, mid + half]
def build_figure() -> go.Figure:
    """Build one panel per available method of the current view, in 2D (lasso) or 3D (orbit)."""
    d, methods, title, n = state["dim"], state["methods"], state["display_title"], len(state["ids"])
    titles = [f"{METHODS[m]}" for m in methods]
    if d == 2:
        fig = make_subplots(rows=1, cols=len(methods), subplot_titles=titles, horizontal_spacing=0.07)
        axis = dict(tickfont=dict(size=FONT_TICK, family=FONT_FAMILY), ticks="outside", linecolor=LINE_COLOR, linewidth=LINE_WIDTH, showgrid=False, zeroline=False, mirror=False)
        for col, m in enumerate(methods, 1):
            c = state["coords"][m]
            fig.add_trace(go.Scattergl(x=c[:, 0], y=c[:, 1], mode="markers", marker=dict(size=MARKER_SIZE, color=[DEFAULT_COLOR] * n, opacity=ALPHA), customdata=state["ids"], hoverinfo="text", name=f"{METHODS[m]} data", showlegend=False), row=1, col=col)
            fig.update_xaxes(title_text=f"{METHODS[m]}-1", range=zoom_range(c[:, 0], ZOOM[m]), **axis, row=1, col=col)
            fig.update_yaxes(title_text=f"{METHODS[m]}-2", range=zoom_range(c[:, 1], ZOOM[m]), **axis, row=1, col=col)
    else:
        fig = make_subplots(rows=1, cols=len(methods), specs=[[{"type": "scatter3d"}] * len(methods)], subplot_titles=titles, horizontal_spacing=0.02)
        axis = dict(backgroundcolor="white", showbackground=True, showline=True, linecolor=LINE_COLOR, linewidth=LINE_WIDTH, gridcolor="#E5E5E5", zerolinecolor="#C8C8C8", tickfont=dict(size=FONT_TICK - 2, family=FONT_FAMILY, color="black"))
        for col, m in enumerate(methods, 1):
            c = state["coords"][m]
            fig.add_trace(go.Scatter3d(x=c[:, 0], y=c[:, 1], z=c[:, 2], mode="markers", marker=dict(size=MARKER_SIZE_3D, color=[rgba(DEFAULT_COLOR, ALPHA)] * n, opacity=1.0, line=dict(width=0)), customdata=state["ids"], hoverinfo="text", name=f"{METHODS[m]} data", showlegend=False), row=1, col=col)
            fig.update_layout({f"scene{'' if col == 1 else col}": {f"{ax}axis": dict(title=f"{METHODS[m]}-{i + 1}", **axis) for i, ax in enumerate("xyz")} | {"aspectmode": "cube"}})
    fig.update_layout(title=dict(text=title, x=0.5, font=dict(size=FONT_TITLE + 2, family=FONT_FAMILY)), width=FIGURE_WIDTH, height=FIGURE_HEIGHT, dragmode="lasso" if d == 2 else "orbit", hovermode="closest", plot_bgcolor="white", paper_bgcolor="white", margin=dict(l=40, r=20, t=90, b=40), legend=dict(orientation="h", y=-0.12, x=0.5, xanchor="center", yanchor="top", bgcolor="rgba(0,0,0,0)", font=dict(size=FONT_LEGEND), itemsizing="constant"), font=dict(size=FONT_TICK, family=FONT_FAMILY, color="#222222"))
    fig.update_annotations(font=dict(size=FONT_TITLE, family=FONT_FAMILY, color="#111111"))
    return fig
def apply_color_by(_: object = None) -> None:
    """Colour the current view by a saved ``obs`` column, turning its categories into editable clusters."""
    column = color_by_dropdown.value
    if state["obs"] is None or column == "(manual)": return
    with output_area:
        clear_output(wait=True)
        if column == "species":
            values = pd.Series(state["label_source"])
            names = [SPECIES_NAMES.get(s, s) for s in species_order(list(state["obs"]["species"].astype(str).unique()))]
            colors = [SPECIES_COLORS.get(n, PALETTE[i % len(PALETTE)]) for i, n in enumerate(names)]
        else:
            values = state["obs"][column].astype(str).where(state["obs"][column].notna())
            names = list(values.value_counts().index)
            colors = None
        lookup = {n: i + 1 for i, n in enumerate(names)}
        state["cluster_labels"] = np.asarray([lookup.get(v, 0) for v in values], dtype=int)
        state["cluster_info"] = palette_info(names if column == "species" else [f"{column} {n}" for n in names], colors)
        update_plot_traces()
        print(f"[PYTHON-INFO] Coloured by '{column}': {len(names)} groups.")
def load_and_plot(change: dict | None = None) -> None:
    """Load the selected view and draw it."""
    label = select_dropdown.value
    with output_area:
        clear_output(wait=True)
        if not label:
            print("[PYTHON-INFO] Select a view.")
            return
        try:
            obs, coords = load_view(label)
            ids, n = obs["protein_id"].astype(str).to_numpy(), len(obs)
            sources = obs["species"].astype(str).map(lambda s: SPECIES_NAMES.get(s, s)).to_numpy()
            state.update(ids=ids, coords=coords, methods=tuple(coords), obs=obs, label_source=sources, current=view_tag(label), dim=VIEWS[label][2], display_title=label.replace(" | ", " · "), selected_indices=[], cluster_labels=np.zeros(n, dtype=int), cluster_info=palette_info([]), filtered_mask=np.ones(n, dtype=bool))
            fit_on_dropdown.options = list(coords)
            mcs_slider.max = max(10, n // 2)
            mcs_slider.value = min(MIN_CLUSTER_SIZE, max(5, n // 20))
            update_checklist(list(dict.fromkeys(SPECIES_NAMES.get(s, s) for s in species_order(list(obs["species"].astype(str).unique())))))
            try: fw = go.FigureWidget(build_figure())
            except Exception as exc: raise RuntimeError("FigureWidget could not initialise: run `pip install -U plotly anywidget ipywidgets` and restart the kernel.") from exc
            for i, m in enumerate(coords):
                fw.data[i].on_selection(selection_callback(m))
                fw.data[i].on_click(selection_callback(m))
            state["fig_widget"] = fw
            plot_container.children = [fw]
            update_plot_traces()
            apply_color_by()
            print(f"[PYTHON-INFO] Loaded {n:,} points for '{label}'. " + ("Lasso/box select points." if state["dim"] == 2 else "Click points to add them to the selection."))
        except Exception as exc:
            print(f"[PYTHON-ERROR] {exc}")
            traceback.print_exc()
def fit_clusters(_: object = None) -> None:
    """Fit HDBSCAN on the chosen layout of the current view."""
    with output_area:
        clear_output(wait=True)
        if state["ids"] is None:
            print("[PYTHON-INFO] Load a view first.")
            return
        method = fit_on_dropdown.value
        try:
            print(f"[PYTHON-INFO] Fitting HDBSCAN on {METHODS[method]} {state['dim']}D (MinCS={mcs_slider.value}, MinSamples={ms_slider.value})...")
            labels, n_clusters = fit_hdbscan_auto(state["coords"][method], mcs_slider.value, ms_slider.value)
        except Exception as exc:
            print(f"[PYTHON-ERROR] {exc}")
            return
        prefix = "U" if method == "umap" else "T"
        state["cluster_labels"], state["cluster_info"] = labels, palette_info([f"{prefix}-{cid:02d}" for cid in range(1, n_clusters + 1)])
        color_by_dropdown.value = "(manual)"
        update_plot_traces()
        clear_output(wait=True)
        print(f"[PYTHON-INFO] {METHODS[method]}: {n_clusters} cluster(s) found, {int(np.sum(labels == 0)):,} points unassigned.")
###################################### EXPORTS
def output_directory() -> Path:
    """Create and return the export directory, relative paths being taken from the project root."""
    raw = plot_fp_text.value.strip()
    if not raw: raise ValueError("Specify an export directory.")
    path = Path(raw) if Path(raw).is_absolute() else PROJECT_ROOT / raw
    path.mkdir(parents=True, exist_ok=True)
    return path
def legend_handles() -> list[Line2D]:
    """Build the legend handles of the current clusters, largest first and unassigned last."""
    labels = state["cluster_labels"]
    order = sorted((cid for cid in state["cluster_info"] if cid != 0 and np.any(labels == cid)), key=lambda c: -int(np.sum(labels == c)))
    handles = [Line2D([0], [0], marker="o", linestyle="", color=state["cluster_info"][c]["color"], label=f"{state['cluster_info'][c]['name']} (n={int(np.sum(labels == c)):,})", markersize=8) for c in order]
    if np.any(labels == 0): handles.append(Line2D([0], [0], marker="o", linestyle="", color=DEFAULT_COLOR, label=f"unassigned (n={int(np.sum(labels == 0)):,})", markersize=8))
    return handles
def export_results(_: object = None) -> None:
    """Export the cluster labels of the current view (selection flagged) to TSV."""
    with output_area:
        clear_output(wait=True)
        if state["ids"] is None:
            print("[PYTHON-INFO] Load a view first.")
            return
        selected = np.zeros(len(state["ids"]), dtype=bool)
        if state["selected_indices"]: selected[state["selected_indices"]] = True
        else: selected = state["filtered_mask"]
        df = pd.DataFrame({"id": state["ids"], "species": state["label_source"], "cluster_id": state["cluster_labels"], "cluster_name": [cluster_name_for(x) for x in state["cluster_labels"]], "view": state["current"], "in_selection": selected})
        path = output_directory() / f"{state['current']}_clusters.tsv"
        df.to_csv(path, sep="\t", index=False)
        print(f"[PYTHON-INFO] Exported {len(df):,} rows to {path}")
def export_html(_: object = None) -> None:
    """Save the current interactive figure as a standalone HTML file."""
    with output_area:
        clear_output(wait=True)
        if state["fig_widget"] is None:
            print("[PYTHON-INFO] Load a view first.")
            return
        path = output_directory() / f"{state['current']}_plot.html"
        go.Figure(state["fig_widget"].to_dict()).write_html(path, include_plotlyjs="cdn")
        print(f"[PYTHON-INFO] Saved {path}")
def export_plot(_: object = None) -> None:
    """Export the current view (matplotlib in 2D, Kaleido in 3D) and its horizontal and vertical legends."""
    with output_area:
        clear_output(wait=True)
        if state["fig_widget"] is None:
            print("[PYTHON-INFO] Load a view first.")
            return
        out_dir, tag, labels, methods = output_directory(), state["current"], state["cluster_labels"], state["methods"]
        colors = np.asarray([cluster_color_for(x) for x in labels])
        order = np.argsort(labels != 0, kind="stable")
        if state["dim"] == 2:
            fig, axes = plt.subplots(1, len(methods), figsize=(SQUARE_SIZE * len(methods) / 2, SQUARE_SIZE / 2), dpi=DPI, squeeze=False)
            for ax, m in zip(axes[0], methods):
                c = state["coords"][m][order]
                ax.scatter(c[:, 0], c[:, 1], c=colors[order], s=MARKER_SIZE, alpha=ALPHA, linewidths=0, rasterized=True)
                ax.set_xlim(*zoom_range(c[:, 0], ZOOM[m]))
                ax.set_ylim(*zoom_range(c[:, 1], ZOOM[m]))
                ax.set_title(METHODS[m], fontsize=FONT_TITLE)
                ax.set_xlabel(f"{METHODS[m]}-1", fontsize=FONT_TICK)
                ax.set_ylabel(f"{METHODS[m]}-2", fontsize=FONT_TICK)
                ax.tick_params(axis="both", labelsize=FONT_TICK - 2, length=4, width=LINE_WIDTH)
                ax.spines[["top", "right"]].set_visible(False)
            fig.suptitle(state["display_title"], fontsize=FONT_TITLE + 2)
            fig.tight_layout()
            handles = legend_handles()
            fig.legend(handles=handles, loc="upper center", ncol=min(LEGEND_NCOL_MAX, max(1, len(handles))), frameon=False, fontsize=FONT_LEGEND, bbox_to_anchor=(0.5, 0.0))
            for ext in ("pdf", "png"): fig.savefig(out_dir / f"{tag}_plot.{ext}", dpi=DPI, bbox_inches="tight", facecolor="white")
            plt.close(fig)
        else:
            fig = go.Figure(state["fig_widget"].to_dict())
            fig.update_layout(width=1600, height=900)
            try:
                for ext in ("pdf", "png"): fig.write_image(str(out_dir / f"{tag}_plot.{ext}"), scale=2 if ext == "png" else 1)
            except Exception as exc:
                print(f"[PYTHON-ERROR] Static 3D export needs Kaleido (`pip install -U kaleido`); use Export HTML instead: {exc}")
                return
        handles = legend_handles()
        for orientation in ("horizontal", "vertical"):
            ncol = min(LEGEND_NCOL_MAX, max(1, len(handles))) if orientation == "horizontal" else 1
            legend_fig = plt.figure(figsize=(12, 0.5 + 0.4 * np.ceil(len(handles) / ncol)) if orientation == "horizontal" else (4, max(2, len(handles) * 0.4)), dpi=DPI)
            legend_fig.legend(handles=handles, loc="center", ncol=ncol, frameon=False, fontsize=FONT_LEGEND)
            for ext in ("pdf", "png"): legend_fig.savefig(out_dir / f"{tag}_legend_{orientation}.{ext}", dpi=DPI, bbox_inches="tight", facecolor="white")
            plt.close(legend_fig)
        print(f"[PYTHON-INFO] Saved plot and legends to {out_dir}")
###################################### BATCH
def run_all(_: object = None) -> None:
    """Fit HDBSCAN on every view whose layouts include the chosen method and export one TSV per view, skipping existing ones unless Overwrite is ticked."""
    with output_area:
        clear_output(wait=True)
        out_dir, method, done, skipped = output_directory(), fit_on_dropdown.value, 0, 0
        todo = [label for label, (*_, methods) in VIEWS.items() if method in methods]
        for label in todo:
            path = out_dir / f"{view_tag(label)}_{method}_hdbscan_mcs{mcs_slider.value}_ms{ms_slider.value}_clusters.tsv"
            if path.exists() and not overwrite_cb.value:
                skipped += 1
                print(f"[PYTHON-INFO] {path.name} already exists. Skipping.")
                continue
            try:
                obs, coords = load_view(label)
                labels, n_clusters = fit_hdbscan_auto(coords[method], mcs_slider.value, ms_slider.value)
                names = palette_info([f"{'U' if method == 'umap' else 'T'}-{cid:02d}" for cid in range(1, n_clusters + 1)])
                pd.DataFrame({"id": obs["protein_id"], "species": obs["species"], "cluster_id": labels, "cluster_name": [names[int(x)]["name"] for x in labels], "view": view_tag(label)}).to_csv(path, sep="\t", index=False)
                done += 1
                print(f"[PYTHON-INFO] [{done}/{len(todo)}] {label}: {n_clusters} clusters")
            except Exception as exc:
                print(f"[PYTHON-ERROR] {label}: {exc}")
        print(f"[PYTHON-INFO] Batch complete: {done} written, {skipped} skipped, {len(todo) - done - skipped} failed.")

In [ ]:
###################################### BINDINGS_AND_DISPLAY
select_dropdown.observe(load_and_plot, names="value")
color_by_dropdown.observe(apply_color_by, names="value")
reset_angles_btn.on_click(reset_angles)
fit_btn.on_click(fit_clusters)
assign_btn.on_click(assign_selected)
clear_sel_btn.on_click(clear_selection)
rename_btn.on_click(rename_cluster)
reset_btn.on_click(reset_clusters)
delete_btn.on_click(delete_cluster)
export_tsv_btn.on_click(export_results)
export_plot_btn.on_click(export_plot)
export_html_btn.on_click(export_html)
run_all_btn.on_click(run_all)
display(VBox([HBox([select_dropdown, color_by_dropdown, reset_angles_btn]), checklist_box, HBox([mcs_slider, ms_slider, fit_on_dropdown, fit_btn]), HBox([cluster_name, cluster_color, assign_btn, clear_sel_btn]), HBox([rename_dropdown, rename_text, rename_btn, delete_btn, reset_btn]), HBox([plot_fp_text, export_tsv_btn, export_plot_btn, export_html_btn, run_all_btn, overwrite_cb]), plot_container, output_area]))
print(f"[PYTHON-INFO] {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} {time.time() - start_time:.2f}s {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024:.2f}MB")